In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np

WIDTH  = 160
HEIGHT = 120
PIXELS = WIDTH * HEIGHT

def compare_two_npy(npy_u16_path, npy_u8_path):
    # A: 串口脚本保存的
    a = np.load(npy_u16_path)

    # 处理 shape
    if a.ndim == 3:
        a = a[0]
    elif a.ndim != 2:
        raise ValueError("Unexpected u16 npy shape")

    if a.shape != (HEIGHT, WIDTH):
        raise ValueError(f"Unexpected u16 shape {a.shape}")

    # B: hexdump 保存的
    b_u8 = np.load(npy_u8_path)

    if b_u8.dtype != np.uint8:
        raise ValueError("Expected uint8 npy")

    if b_u8.size != PIXELS * 2:
        raise ValueError(
            f"Size mismatch: {b_u8.size} bytes vs {PIXELS*2}"
        )

    # u8 → u16（大端）
    b = b_u8.view(">u2").reshape((HEIGHT, WIDTH))

    # 真正的对比
    equal = np.array_equal(a, b)

    print(f"Pixel-by-pixel equal: {equal}")

    if not equal:
        diff = np.abs(a.astype(np.int32) - b.astype(np.int32))
        max_diff = diff.max()
        idx = np.unravel_index(diff.argmax(), diff.shape)

        print(f"Max diff: {max_diff}")
        print(f"First mismatch at {idx}:")
        print(f"  A(u16) = {a[idx]}")
        print(f"  B(u16) = {b[idx]}")
    else:
        print("✅ Raw data is IDENTICAL")


# 用法示例
compare_two_npy(
    "capture_20260124_102321/frames.npy",   # 串口脚本生成
    "capture_20260124_102321/data14.npy"         # hexdump 脚本生成
)


Pixel-by-pixel equal: False
Max diff: 63884
First mismatch at (107, 19):
  A(u16) = 65028
  B(u16) = 1144
